# 5-02 지표·코드 찾기 — "이 지표 DeepSearch에 있어? 무슨 코드로 봐?"

**질문 예시:** "삼성전자 종목코드 알려줘", "코스피 지수 코드 뭐야?", "원/달러 환율은 어떤 코드로 봐?",
"베이시스 데이터 있어?", "신용융자 잔고 어디서 봐?", "한국 관련 FRED 지표 뭐 있어?"

CHECK 원본(`5_데이터조회/02_지표코드찾기`)은 코드 카탈로그와 응답 필드까지 뒤져 "어디서 보는지"를 찾았습니다.
DeepSearch에는 시리즈 검색 함수 `FindEntity("DataSeries", 검색어)`가 있지만 **어떤 검색어든 0건**이라 쓸 수 없습니다(2026-11-25 확인).
대신 아래 방법을 씁니다.

| 찾는 것 | 방법 | 함수 |
|---|---|---|
| 종목코드 | 전 종목 목록에서 이름 검색 | `GetStockSymbols` |
| 지수 코드 | 지수 목록(11종) | `GetMarketIndexSymbols` |
| FRED 지표 | 아는 시리즈 → 속한 카테고리 → 같은 카테고리 시리즈 목록 | `GetFredCategoryOfDataSeries`, `GetFredDataSeriesListUnderCategory`, `QueryFredSeriesInfo` |
| 한국은행 지표 | 통계표 코드로 항목·**마지막 시점** 확인 | `QueryBankOfKoreaSeriesInfo` |
| IMF·OECD 지표 | 한 시리즈에서 같은 국가·같은 지표 목록 펼치기 | `GetImfIndicatorListOfSameCountry`, `GetOecdRelatedIndicatorList` |
| 시세·밸류 지표 이름 | 자연어 질의가 데이터로 해석되는지 확인 | compute 자연어 |

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beaten-by-the-market/deepsearch/blob/main/scenarios/5_data_lookup/02_find_indicator.ipynb)

## 0. 준비
Colab 왼쪽 🔑(Secrets)에 `DEEPSEARCH_API_KEY`를 등록하고 이 노트북의 액세스를 켠 뒤 아래 두 셀을 실행하세요. 두 번째 셀은 모든 시나리오에 공통인 헬퍼입니다(접어 두어도 됩니다).

In [1]:
# Colab에서만: 한글 차트 폰트 설치 (로컬에서는 건너뜀)
import sys
if "google.colab" in sys.modules:
    !pip -q install koreanize-matplotlib

In [2]:
# ── DeepSearch 헬퍼 (모든 시나리오 노트북 공통) ─────────────────────────────
# Colab: 왼쪽 🔑(Secrets)에 이름 DEEPSEARCH_API_KEY 로 키를 넣고 "노트북 액세스"를 켜세요.
#        키는 "아이디:비밀번호" 형식이든 base64 문자열이든 됩니다. 키 값은 출력하지 않습니다.
# 로컬: 환경변수 DEEPSEARCH_API_KEY / API_KEY 또는 상위 폴더의 .env 를 읽습니다.
import base64, json, os, sys, time
from datetime import date, datetime, timedelta

import pandas as pd
import requests

IN_COLAB = "google.colab" in sys.modules
API_URL = "https://api.deepsearch.com/v1/compute"

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)


def _norm_key(k):
    k = k.strip().strip('"').strip("'")
    return base64.b64encode(k.encode()).decode() if ":" in k else k


def _find_key():
    if IN_COLAB:
        try:
            from google.colab import userdata
            k = userdata.get("DEEPSEARCH_API_KEY")
            if k:
                return _norm_key(k)
        except Exception:
            pass
    for name in ("DEEPSEARCH_API_KEY", "API_KEY"):
        if os.environ.get(name):
            return _norm_key(os.environ[name])
    d = os.path.abspath(os.getcwd())
    while True:
        p = os.path.join(d, ".env")
        if os.path.isfile(p):
            for line in open(p, encoding="utf-8-sig"):
                if "=" in line and not line.strip().startswith("#"):
                    k, v = line.split("=", 1)
                    if k.strip() in ("DEEPSEARCH_API_KEY", "API_KEY") and v.strip():
                        return _norm_key(v)
        if os.path.dirname(d) == d:
            break
        d = os.path.dirname(d)
    from getpass import getpass
    return _norm_key(getpass("DeepSearch API 키: "))


_KEY = _find_key()


class DSError(Exception):
    pass


def ds(query, timeout=180, retries=3):
    """compute 호출 → 응답 JSON. 서버가 돌려준 오류(data.exceptions)는 DSError."""
    last = None
    for i in range(retries):
        try:
            r = requests.post(API_URL, json={"input": query}, timeout=timeout,
                              headers={"Authorization": "Basic " + _KEY})
            body = r.json()
        except Exception as e:  # 네트워크·일시 오류만 재시도
            last = e
            time.sleep(3 * (i + 1))
            continue
        exc = (body.get("data") or {}).get("exceptions") or []
        if exc or r.status_code >= 400:
            msg = "; ".join(f"{x.get('class', '')}: {' '.join(map(str, x.get('args') or [x.get('message', '')]))}"
                            if isinstance(x, dict) else str(x) for x in exc)
            raise DSError(f"{r.status_code} {msg or body}"[:500])
        return body
    raise DSError(f"요청 실패: {last}")


def _pod(body):
    pods = (body.get("data") or {}).get("pods") or []
    return pods[1] if len(pods) >= 2 else {}


def ds_df(query, **kw):
    """DataFrame 결과(재무·시세·경제지표 등) → pandas DataFrame. 결과가 없으면 빈 DataFrame."""
    c = _pod(ds(query, **kw)).get("content") or {}
    data = c.get("data") if isinstance(c, dict) else None
    if not isinstance(data, dict) or not data:
        return pd.DataFrame()
    cols = [x for x in (c.get("columns") or data.keys()) if x in data]
    df = pd.DataFrame({x: data[x] for x in cols})
    for x in ("date", "created_at"):
        if x in df and df[x].dtype == object:
            df[x] = pd.to_datetime(df[x].astype(str).str[:19], errors="coerce")
    return df


def _polarity(x):
    """문서 긍부정: polarity.name, 없으면 customized_ml.kis 모델 값(최근 뉴스는 절반가량이 이쪽에만 있음)."""
    p = x.get("polarity")
    if isinstance(p, dict) and p.get("name"):
        return p["name"]
    k = (((x.get("customized_ml") or {}).get("kis") or {}).get("ip_multitask") or {}).get("polarity") or {}
    return k.get("name")


def doc_search(category, section, query, date_from, date_to, count=100, pages=1):
    """DocumentSearch → (총건수, 문서 DataFrame). category: news/research/company/patent. 기간은 1년 이내.
    종목 지정은 securities.symbol:005930 (접두어 없는 6자리)이 뉴스·리서치·공시 모두에서 가장 정확합니다.
    securities.symbol:KRX:005930 은 키워드 검색으로 바뀌어 건수가 줄고, securities.name: 은 리서치(공식명
    '에스케이하이닉스')·공시(DART명 '현대자동차')에서 놓칩니다(2026-11-25 확인). 여러 종목: securities.symbol:(005930 or 000660)"""
    rows, total = [], 0
    for page in range(1, pages + 1):
        q = (f'DocumentSearch(["{category}"],{json.dumps(section, ensure_ascii=False)},'
             f'{json.dumps(query, ensure_ascii=False)},count={count},page={page},'
             f'date_from={date_from},date_to={date_to})')
        c = _pod(ds(q)).get("content") or {}
        d = c.get("data") if isinstance(c.get("data"), dict) else c
        total = d.get("total_matches") or 0
        docs = d.get("docs") or []
        for x in docs:
            secs = x.get("securities") or []
            rows.append({
                "created_at": pd.to_datetime(str(x.get("created_at"))[:19], errors="coerce"),
                "publisher": x.get("publisher"),
                "title": " ".join(str(x.get("title") or "").split()),
                "stocks": ", ".join(s.get("name", "") for s in secs),
                "symbols": ", ".join(s.get("symbol", "") for s in secs),
                "polarity": _polarity(x),
                "url": x.get("content_url"),
            })
        if len(docs) < count:
            break
    cols = ["created_at", "publisher", "title", "stocks", "symbols", "polarity", "url"]
    return total, pd.DataFrame(rows, columns=cols)


def doc_trends(category, section, query, date_from, date_to, interval="1d"):
    """DocumentTrends → DataFrame(date, count)."""
    q = (f'DocumentTrends(["{category}"],{json.dumps(section, ensure_ascii=False)},'
         f'{json.dumps(query, ensure_ascii=False)},interval="{interval}",'
         f'date_from={date_from},date_to={date_to})')
    c = _pod(ds(q)).get("content") or {}
    d = c.get("data") if isinstance(c.get("data"), dict) else c
    trends = d.get("trends") or []
    b = trends[0].get("buckets", []) if trends else []
    df = pd.DataFrame([{"date": x.get("key_as_string") or x.get("key"), "count": x.get("count", 0)} for x in b])
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"].astype(str).str[:19], errors="coerce")
    return df


def stock_prices(entities, date_from, date_to, columns=None):
    """종목 목록의 일별 시세. entities 예: ["KRX:005930", "KRX:000660"] 또는 "KRX:005930".
    가격은 수정주가, 거래량·거래대금·시가총액은 당일 실제 값. change_rate 는 비율(0.01 = 1%)."""
    ents = [entities] if isinstance(entities, str) else list(entities)
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    q = (f'GetStockPrices([{", ".join(ents)}], columns={json.dumps(cols)}, '
         f'date_from={date_from}, date_to={date_to})')
    return ds_df(q)


_UNIVERSE = None


def universe():
    """현재 상장 종목 기본정보(GetStockSymbols 전 종목): symbol, name, market, industry_name 등.
    company_type_l1: 1 일반법인, 2 공기업(한국전력·기업은행 등 8종목), 8 ETF·ETN 등."""
    global _UNIVERSE
    if _UNIVERSE is None:
        u = ds_df("GetStockSymbols()")
        _UNIVERSE = u.rename(columns={"entity_name": "name"})[
            ["symbol", "name", "market", "company_type_l1", "industry_id", "industry_name", "date_listed",
             "fiscal_year_end", "conglomerate_id"]]
    return _UNIVERSE.copy()


def all_stocks(day, common_only=True, columns=None):
    """day(또는 그 이전 마지막 거래일)의 전 종목 하루치 시세 + 종목 정보. 한 번 호출로 4,000여 종목.
    common_only=True 이면 일반법인·공기업 보통주만(ETF·ETN·우선주 제외)."""
    day = pd.Timestamp(day).date()
    cols = columns or ["open", "high", "low", "close", "volume", "value", "change_rate", "market_cap"]
    snap = ds_df(f'GetStockPrices(columns={json.dumps(cols)}, '
                 f'date_from={day - timedelta(days=7)}, date_to={day})')
    df = snap.merge(universe().drop(columns=["date"], errors="ignore"), on="symbol", how="left")
    if common_only:
        df = df[df["company_type_l1"].isin(["1", "2"]) & df["symbol"].str.endswith("0")]
    df = df[df["market"].isin(["KOSPI", "KOSDAQ", "KONEX"])]
    if "change_rate" in df:
        df["change_pct"] = df["change_rate"] * 100
    return df.reset_index(drop=True)


def index_prices(date_from, date_to):
    """지수 11종(코스피·코스닥 계열) 일별. change_rate 는 퍼센트(1.94 = 1.94%).
    지수의 value(거래대금)는 백만원, volume(거래량)은 천주 단위로 보입니다(코스닥 2026-11-23 value 4,673,819 ≈ 4.7조)."""
    # 지수를 지정하지 않으면 최근 약 1,300행(≈118거래일)만 오고 앞 기간이 조용히 잘린다 → 140일씩 나눠 받는다
    start, end = pd.Timestamp(date_from), pd.Timestamp(date_to)
    parts = []
    while start <= end:
        stop = min(start + pd.Timedelta(days=139), end)
        parts.append(ds_df(f"GetMarketIndexes(date_from={start.date()}, date_to={stop.date()})"))
        start = stop + pd.Timedelta(days=1)
    df = pd.concat([x for x in parts if not x.empty], ignore_index=True) if parts else pd.DataFrame()
    return df.drop_duplicates(["date", "symbol"]).sort_values(["date", "symbol"]).reset_index(drop=True) if not df.empty else df


def fred(series, date_from, date_to):
    """FRED 시리즈 → 날짜 × 시리즈 넓은 표. 예: fred(["DGS10", "DEXKOUS"], "2026-10-01", "2026-11-24")"""
    ids = ", ".join(json.dumps(s) for s in series)
    df = ds_df(f"QueryFredSeriesData([{ids}], date_from={date_from}, date_to={date_to})")
    if df.empty:
        return df
    val = [c for c in df.columns if c not in ("entity_id", "date")][0]
    df["series"] = df["entity_id"].str.replace("FRED:", "", regex=False)
    wide = df.pivot_table(index="date", columns="series", values=val).sort_index()
    return wide[[s for s in series if s in wide.columns]]  # 요청한 순서대로


def krw(v, digits=1):
    """원 → '138.0조' / '5,027억'."""
    if v is None or pd.isna(v):
        return "-"
    a = abs(v)
    if a >= 1e12:
        return f"{v / 1e12:.{digits}f}조"
    if a >= 1e8:
        return f"{v / 1e8:,.0f}억"
    if a >= 1e4:
        return f"{v / 1e4:,.0f}만"
    return f"{v:,.0f}"


PRICE_NOTICE = ("※ 주가는 수정주가입니다(분할·무상증자·유상증자 소급 반영, 배당 미반영). "
                "거래량·거래대금·시가총액은 당일 실제 값이며 KRX 기준입니다(NXT 미포함).")

# 한글 차트 폰트
import matplotlib.pyplot as plt
try:
    import koreanize_matplotlib  # noqa: F401  (Colab: 위 셀에서 설치)
except ImportError:
    if sys.platform.startswith("win"):
        plt.rcParams["font.family"] = "Malgun Gothic"
    elif sys.platform == "darwin":
        plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

print("DeepSearch 준비 완료", "(Colab)" if IN_COLAB else "(로컬)")

DeepSearch 준비 완료 (로컬)


In [3]:
ASOF = "2026-11-24"

## 1. 종목코드 찾기
`universe()`는 `GetStockSymbols()` 전 종목(4,400여 개, ETF 포함) 표입니다. 이름 일부로 찾습니다.

In [4]:
u = universe()

def find_stock(text):
    hit = u[u["name"].str.contains(text, case=False, regex=False)]
    return hit[["symbol", "name", "market", "company_type_l1", "industry_name"]].head(10)

display(find_stock("삼성전자"))
find_stock("하이닉스")

,symbol,name,market,company_type_l1,industry_name
400,KRX:005930,삼성전자,KOSPI,1,이동전화기 제조업
401,KRX:005935,삼성전자우,KOSPI,1,이동전화기 제조업
804,KRX:0162Z0,RISE 삼성전자SK하이닉스채권혼합50,KOSPI,8,None
850,KRX:0177N0,KODEX 삼성전자SK하이닉스채권혼합50,KOSPI,8,None
873,KRX:0183V0,KIWOOM 삼성전자&SK하이닉스채권혼합50,KOSPI,8,None
904,KRX:0192M0,RISE 삼성전자단일종목레버리지,KOSPI,8,None
909,KRX:0193K0,PLUS 삼성전자단일종목레버리지,KOSPI,8,None
910,KRX:0193L0,PLUS 삼성전자선물단일종목인버스2X,KOSPI,8,None
913,KRX:0193W0,KODEX 삼성전자단일종목레버리지,KOSPI,8,None
915,KRX:0194M0,ACE 삼성전자단일종목레버리지,KOSPI,8,None


,symbol,name,market,company_type_l1,industry_name
56,KRX:000660,SK하이닉스,KOSPI,1,"다이오드, 트랜지스터 및 유사 반도체소자 제조업"
804,KRX:0162Z0,RISE 삼성전자SK하이닉스채권혼합50,KOSPI,8,None
850,KRX:0177N0,KODEX 삼성전자SK하이닉스채권혼합50,KOSPI,8,None
873,KRX:0183V0,KIWOOM 삼성전자&SK하이닉스채권혼합50,KOSPI,8,None
903,KRX:0192L0,RISE SK하이닉스단일종목레버리지,KOSPI,8,None
912,KRX:0193T0,KODEX SK하이닉스단일종목레버리지,KOSPI,8,None
917,KRX:0194R0,KIWOOM SK하이닉스선물단일종목레버리지,KOSPI,8,None
918,KRX:0194T0,ACE SK하이닉스단일종목레버리지,KOSPI,8,None
923,KRX:0195S0,TIGER SK하이닉스단일종목레버리지,KOSPI,8,None
929,KRX:0197W0,SOL SK하이닉스단일종목레버리지,KOSPI,8,None


## 2. 지수 코드
DeepSearch 지수는 코스피·코스닥 계열 11종뿐입니다. KRX 섹터지수·테마지수·KRX300·VKOSPI는 없습니다.

In [5]:
ds_df("GetMarketIndexSymbols()")[["symbol", "name_ko", "name_en", "index_id"]]

,symbol,name_ko,name_en,index_id
0,KRX:KOSPI,코스피,KOSPI,001
1,KRX:KOSPILC,코스피 대형주,KOSPI Large Cap,002
2,KRX:KOSPIMC,코스피 중형주,KOSPI Mid Cap,003
3,KRX:KOSPISC,코스피 소형주,KOSPI Small Cap,004
4,KRX:KOSPI200,코스피 200,KOSPI 200,029
5,KRX:KOSPI100,코스피 100,KOSPI 100,035
6,KRX:KOSPI50,코스피 50,KOSPI 50,036
7,KRX:KOSDAQ,코스닥,KOSDAQ,001
8,KRX:KOSDAQLC,코스닥 대형주,KOSDAQ Large Cap,002
9,KRX:KOSDAQMC,코스닥 중형주,KOSDAQ Mid Cap,003


## 3. FRED 지표 찾기 — 카테고리 따라가기
아는 시리즈 하나에서 출발해 그 시리즈가 속한 카테고리를 보고, 같은 카테고리의 다른 시리즈를 펼칩니다.
펼친 목록을 제목 키워드로 거르고, **마지막 관측일**로 지금도 갱신되는지 확인합니다.
예: 한국 10년 금리(`IRLTLT01KRM156N`)와 원/달러(`DEXKOUS`)에서 출발.

In [6]:
cats = ds_df('GetFredCategoryOfDataSeries(["IRLTLT01KRM156N", "DEXKOUS"])')
cats

,entity_id,category_id,name
0,FRED:IRLTLT01KRM156N,32286,"Korea, Republic of (South Korea)"
1,FRED:IRLTLT01KRM156N,32949,Interest Rates
2,FRED:DEXKOUS,94,Daily Rates
3,FRED:DEXKOUS,261,South Korea


In [7]:
def fred_category(cat_ids, keyword=None, fresh_after="2026-01-01"):
    ids = ", ".join(str(int(c)) for c in cat_ids)
    df = ds_df(f"GetFredDataSeriesListUnderCategory([{ids}])")
    if keyword:
        df = df[df["title"].str.contains(keyword, case=False, regex=True)]
    df = df.assign(최신=pd.to_datetime(df["observation_end"]) >= pd.Timestamp(fresh_after))
    return df[["entity_id", "title", "frequency_short", "observation_end", "최신"]].sort_values(
        ["최신", "observation_end"], ascending=False).drop_duplicates("entity_id").reset_index(drop=True)

korea_cat = cats.loc[cats["name"].str.startswith("Korea, Republic"), "category_id"].unique().tolist()
kr = fred_category(korea_cat)
print(f"'Korea, Republic of' 카테고리 시리즈 {len(kr)}개, 이 중 2026년까지 갱신되는 것 {kr['최신'].sum()}개")
kr[kr["최신"]].head(20)

'Korea, Republic of' 카테고리 시리즈 2395개, 이 중 2026년까지 갱신되는 것 729개


,entity_id,title,frequency_short,observation_end,최신
0,FRED:GGGDTPKRA188N,Projection of General government gross debt for Korea,A,2029-01-01,True
1,FRED:GGNLBPKRA188N,Projection of General government net lending/borrowing f...,A,2029-01-01,True
2,FRED:KORBCAGDPBP6PT,Balance of Payments: Total Net Current Account for Repub...,A,2029-01-01,True
3,FRED:KORGGXCNLG01GDPPT,Fiscal Situation of General Government: Net Lending/borr...,A,2029-01-01,True
4,FRED:KORNGDPRPCPPPT,National Accounts: Real Gross Domestic Product for Repub...,A,2029-01-01,True
5,FRED:KORPCPIPCPPPT,Prices: Consumer Price Index for Republic of Korea,A,2029-01-01,True
6,FRED:BSCICP02KRM460S,Business Tendency Surveys (Manufacturing): Confidence In...,M,2026-10-01,True
7,FRED:BSOITE02KRM460S,Business Tendency Surveys: Orders Inflow: Economic Activ...,M,2026-10-01,True
8,FRED:BSPRFT02KRM460S,Business Tendency Surveys: Production: Economic Activity...,M,2026-10-01,True
9,FRED:BSPRTE02KRM460S,Business Tendency Surveys: Production: Economic Activity...,M,2026-10-01,True


키워드로 좁히기: 같은 카테고리에서 금리(`Interest|Yield`)만, 일별 환율 카테고리(`Daily Rates`)에서 원·엔·위안만 봅니다.

In [8]:
display(fred_category(korea_cat, keyword="Interest|Yield"))
daily_fx = cats.loc[cats["name"].eq("Daily Rates"), "category_id"].tolist()
fred_category(daily_fx, keyword="Korea|Japan|China")

,entity_id,title,frequency_short,observation_end,최신
0,FRED:IR3TIB01KRM156N,Interest Rates: 3-Month or 90-Day Rates and Yields: Inte...,M,2026-10-01,True
1,FRED:IRLTLT01KRM156N,Interest Rates: Long-Term Government Bond Yields: 10-Yea...,M,2026-10-01,True
2,FRED:IRSTCI01KRM156N,Interest Rates: Immediate Rates (< 24 Hours): Call Money...,M,2026-10-01,True
3,FRED:IR3TIB01KRQ156N,Interest Rates: 3-Month or 90-Day Rates and Yields: Inte...,Q,2026-07-01,True
4,FRED:IRLTLT01KRQ156N,Interest Rates: Long-Term Government Bond Yields: 10-Yea...,Q,2026-07-01,True
5,FRED:IRSTCI01KRQ156N,Interest Rates: Immediate Rates (< 24 Hours): Call Money...,Q,2026-07-01,True
6,FRED:IR3TIB01KRA156N,Interest Rates: 3-Month or 90-Day Rates and Yields: Inte...,A,2025-01-01,False
7,FRED:IRLTLT01KRA156N,Interest Rates: Long-Term Government Bond Yields: 10-Yea...,A,2025-01-01,False
8,FRED:IRSTCI01KRA156N,Interest Rates: Immediate Rates (< 24 Hours): Call Money...,A,2025-01-01,False
9,FRED:IR3TCD01KRM156N,Interest Rates: 3-Month or 90-Day Rates and Yields: Cert...,M,2023-12-01,False


,entity_id,title,frequency_short,observation_end,최신
0,FRED:DEXJPUS,Japanese Yen to U.S. Dollar Spot Exchange Rate,D,2026-11-20,True
1,FRED:DEXKOUS,South Korean Won to U.S. Dollar Spot Exchange Rate,D,2026-11-20,True


찾은 시리즈의 정의(단위·주기·최근 갱신)는 `QueryFredSeriesInfo`로 확인합니다.

In [9]:
info = ds_df('QueryFredSeriesInfo(["DEXKOUS", "IRLTLT01KRM156N", "DCOILBRENTEU"])')
info[["entity_id", "title", "frequency_short", "units_short", "observation_end", "last_updated"]]

,entity_id,title,frequency_short,units_short,observation_end,last_updated
0,FRED:DEXKOUS,South Korean Won to U.S. Dollar Spot Exchange Rate,D,South Korean Won to 1 U.S. $,2026-11-20,2026-11-23T15:16:29
1,FRED:IRLTLT01KRM156N,Interest Rates: Long-Term Government Bond Yields: 10-Yea...,M,%,2026-10-01,2026-11-16T16:43:09
2,FRED:DCOILBRENTEU,Crude Oil Prices: Brent - Europe,D,$ per Barrel,2026-11-17,2026-11-18T12:18:23


## 4. 한국은행 지표 — 있지만 2024년 4월에 멈춤
통계표 코드를 알면 항목 목록과 **마지막 시점(`end_time`)**을 볼 수 있습니다. 원/달러(`731Y001`)와 증시주변자금(`901Y056`)을 보면
둘 다 2024년 3~4월에서 끝납니다. 한국은행 시리즈를 쓸 땐 반드시 이 값을 먼저 보세요.

In [10]:
def bok_items(stat_code, n=8):
    df = ds_df(f'QueryBankOfKoreaSeriesInfo(stat_codes="{stat_code}")')
    df = df.sort_values("end_time").drop_duplicates("item_code", keep="last").sort_values(["end_time", "item_code"], ascending=[False, True])
    print(f"{df['stat_name_ko'].iloc[0]} — 항목 {len(df)}개, 가장 늦은 마지막 시점: {df['end_time'].max()}")
    return df[["item_code", "item_name_ko", "cycle", "end_time"]].head(n).reset_index(drop=True)

display(bok_items("731Y001", 4))
bok_items("901Y056")

3.1.1.1. 주요국 통화의 대원화환율 — 항목 53개, 가장 늦은 마지막 시점: 20240426


,item_code,item_name_ko,cycle,end_time
0,0000001,원/미국달러(매매기준율),DD,20240426
1,0000002,원/일본엔(100엔),DD,20240426
2,0000003,원/유로,DD,20240426
3,0000012,원/영국파운드,DD,20240426


1.5.1.4. 증시주변자금동향 — 항목 6개, 가장 늦은 마지막 시점: 202403


,item_code,item_name_ko,cycle,end_time
0,S23A,투자자 예탁금,MM,202403
1,S23B,파생상품거래 예수금 1),MM,202403
2,S23C,RP,MM,202403
3,S23D,위탁매매 미수금,MM,202403
4,S23E,신용융자 잔고 2),MM,202403
5,S23F,신용대주 잔고 2),MM,202403


## 5. IMF·OECD — 한 시리즈에서 펼치기
IMF·OECD 시리즈 ID는 `기관:데이터셋.국가.지표` 구조입니다. 시리즈 하나만 알면 같은 국가의 다른 지표,
같은 지표의 다른 국가 목록을 펼칠 수 있습니다. 예: 호주 경상수지(IMF BOP)에서 한국(KR) 지표로 바꾸기.
**주의:** IMF 국제수지(BOP)는 한국 연간 경상수지가 2021년에서 끝나고 분기 시리즈는 비어 있습니다(2026-11-25 확인). IMF·OECD 시리즈도 마지막 날짜를 꼭 보세요.

In [11]:
imf = ds_df('GetImfIndicatorListOfSameCountry("IMF:BOP.A.AU.BCA_BP6_USD")')
imf.columns = ["entity_id", "지표"]
print(f"IMF BOP 데이터셋, 같은 국가 지표 {len(imf):,}개 중 'Current Account' 앞부분:")
display(imf[imf["지표"].str.startswith("Current Account")].head(5))
kr_ca = ds_df('GetImfSeriesValues(["IMF:BOP.A.KR.BCA_BP6_USD"], date_from=2015-01-01)')
kr_ca.columns = ["entity_id", "date", "경상수지(백만 달러)"]
print("국가 코드만 KR로 바꾼 한국 경상수지(연간). 마지막:", kr_ca["date"].max().date())
kr_ca.tail(4)

IMF BOP 데이터셋, 같은 국가 지표 1,842개 중 'Current Account' 앞부분:


,entity_id,지표
3,IMF:BOP.A.AU.BCA_BP6_USD,"Current Account, Total, Net, US Dollars"
4,IMF:BOP.A.AU.BCA_BP6_XDC,"Current Account, Total, Net, National Currency"
627,IMF:BOP.A.AU.BGM_BP6_USD,"Current Account, Goods and Services, Goods, General Merc..."
628,IMF:BOP.A.AU.BGM_BP6_XDC,"Current Account, Goods and Services, Goods, General Merc..."
629,IMF:BOP.A.AU.BGN_BP6_USD,"Current Account, Goods and Services, Goods, Non-monetary..."


국가 코드만 KR로 바꾼 한국 경상수지(연간). 마지막: 2021-12-31


,entity_id,date,경상수지(백만 달러)
3,IMF:BOP.A.KR.BCA_BP6_USD,2018-12-31,77466.5
4,IMF:BOP.A.KR.BCA_BP6_USD,2019-12-31,59676.1
5,IMF:BOP.A.KR.BCA_BP6_USD,2020-12-31,75902.2
6,IMF:BOP.A.KR.BCA_BP6_USD,2021-12-31,88302.2


## 6. "이 지표 있어?" 판정표
시세·밸류 지표는 자연어 질의(`삼성전자 지표 기간`)가 **데이터 컬럼**으로 오는지, 아니면 **기업 개요 표**(`ceo` 등 컬럼)로
빠지는지로 판정합니다. 기업 개요 표로 빠지면 오류 없이 "없음"이라는 뜻입니다.

In [12]:
TERMS = ["종가", "시가총액", "PER", "PBR", "배당수익률", "EPS", "BPS",
         "외국인보유비율", "공매도", "신용잔고", "베이시스", "VKOSPI", "투자자예탁금"]
rows = []
for term in TERMS:
    df = ds_df(f"삼성전자 {term} 2026-11-20-{ASOF}")
    ok = "ceo" not in df.columns and len(df) > 0
    last = df.iloc[-1, -1] if ok else None
    rows.append({"지표": term, "자연어로 나옴": "✅" if ok else "❌ (기업 개요 표로 빠짐)",
                 "삼성전자 최근 값": f"{float(last):,.2f}" if ok and pd.notna(last) else ""})
pd.DataFrame(rows)

,지표,자연어로 나옴,삼성전자 최근 값
0,종가,✅,"189,400.00"
1,시가총액,✅,"1,107,285,168,355,200.00"
2,PER,✅,28.68
3,PBR,✅,2.96
4,배당수익률,✅,0.94
5,EPS,✅,"6,605.00"
6,BPS,✅,"63,997.00"
7,외국인보유비율,❌ (기업 개요 표로 빠짐),
8,공매도,❌ (기업 개요 표로 빠짐),
9,신용잔고,❌ (기업 개요 표로 빠짐),


## 7. 정리 — 자주 묻는 지표의 위치

| 지표 | DeepSearch에서 | 상태 |
|---|---|---|
| 삼성전자 종목코드 | `KRX:005930` (`GetStockSymbols`) | ✅ |
| 코스피 종합 지수 | `KRX:KOSPI` (`GetMarketIndexes`) | ✅ |
| 원/달러 환율 | FRED `DEXKOUS` (일별, 1~3일 늦음) | ✅ — 한국은행 `731Y001.0000001`은 2024-04 중단 |
| 한국 경상수지 | IMF `IMF:BOP.A.KR.BCA_BP6_USD` | ⚠️ 2021년에서 끝남 |
| PER·PBR·배당수익률 | 자연어 `삼성전자 PER PBR 기간`, 전 종목은 `상장 기업 and PER 날짜` | ✅ (CHECK에선 일별 필드 없음) |
| 브렌트유 | FRED `DCOILBRENTEU` | ✅ (CHECK에선 범위 밖) |
| 한국 10년 금리 | FRED `IRLTLT01KRM156N` (OECD, 월평균) | ⚠️ 월별만 |
| 신용융자 잔고 | 한국은행 `901Y056` S23E (시장 합계, 월) | ⚠️ 2024-03 중단, 종목별 없음 |
| 투자자예탁금 | 한국은행 `901Y056` S23A | ⚠️ 2024-03 중단 |
| 베이시스·미결제약정·외국인보유비율·공매도·VKOSPI | 없음 | ❌ (`research/data_gaps`) |